In [1]:
# Import required libraries
import pandas as pd
import re

In [2]:
# Load quantity table - roughly an example medication issue table containing quantities
qty_table_full = pd.read_csv('quantity.csv', dtype={'dmd_id': str})
# Full quantity table contains formatted_quantity and formatted_units - essentially what we are trying to extract. Hide these for now - we can test our results against them later
qty_table = qty_table_full.iloc[:, 0:3]

# Load example codelist - this is a codelist for items in our quantity table. Roughly replicates OpenCodelists format but with an added uom field (taken from dm+d)
codelist = pd.read_csv('codelist.csv', dtype={'dmd_id': str})

### Quantity table
Roughly an example medication issue table containing quantities following formats seen in [OpenSAFELY documentation](https://docs.opensafely.org/ehrql/reference/schemas/raw.tpp/#medications).

The full file `qty_table_full` contains columns formatted_quantity and formatted_units which is what we are aiming to extract - use to test results against. 

In [3]:
qty_table

,dmd_nm,dmd_id,quantity
0,Ramipril 5mg tablets,42381711000001108,28 tablets
1,Ramipril 5mg capsules,42381611000001104,28 capsules
2,Ramipril 5mg tablets,42381711000001108,1 pack of 28 tablet(s)
3,Ramipril 5mg capsules,42381611000001104,2 packs of 28 capsule(s)
4,Salbutamol 100micrograms/dose inhaler CFC free,39113611000001102,600 doses
5,Salbutamol 100micrograms/dose inhaler CFC free,39113611000001102,1 pack of 200 dose(s)
6,Clobetasone 0.05% cream,41894611000001102,1 pack of 100 gram(s)
7,Clobetasone 0.05% cream,41894611000001102,15 gram(s)
8,Hypromellose 0.5% eye drops,42191711000001101,10ml - 0.5%
9,Ferrous fumarate 140mg/5ml oral solution,39108711000001103,100 millilitres


## Codelist

This is a codelist for items in our quantity table. Roughly replicates OpenCodelists format but with an added uom field (taken from dm+d).

Generated using BigQuery:
```
SELECT DISTINCT
  id AS code,
  nm AS term,
  id AS dmd_id,
  "VMP" AS dmd_type,
  uom.descr AS uom
FROM `ebmdatalab.dmd.vmp` vmp
LEFT JOIN `dmd.unitofmeasure` uom ON vmp.udfs_uom = uom.cd
WHERE id IN (
  42381711000001108,
  42381611000001104,
  39113611000001102,
  41894611000001102,
  42191711000001101,
  39108711000001103,
  42109611000001109,
  41900511000001104,
  689211000001105,
  35720011000001107,
  41952111000001102,
  42209311000001101,
  42294811000001108,
  39111511000001100,
  249411000001101
)
```

In [4]:
codelist

,code,term,dmd_id,dmd_type,uom
0,42294811000001108,Furosemide 40mg tablets,42294811000001108,VMP,tablet
1,42381611000001104,Ramipril 5mg capsules,42381611000001104,VMP,capsule
2,42191711000001101,Hypromellose 0.5% eye drops,42191711000001101,VMP,NaN
3,42109611000001109,Paracetamol 500mg tablets,42109611000001109,VMP,tablet
4,42209311000001101,Doxazosin 4mg tablets,42209311000001101,VMP,tablet
5,39113611000001102,Salbutamol 100micrograms/dose inhaler CFC free,39113611000001102,VMP,dose
6,41900511000001104,Hydrocortisone 1% cream,41900511000001104,VMP,NaN
7,41952111000001102,Metronidazole 400mg tablets,41952111000001102,VMP,tablet
8,41894611000001102,Clobetasone 0.05% cream,41894611000001102,VMP,NaN
9,39108711000001103,Ferrous fumarate 140mg/5ml oral solution,39108711000001103,VMP,ml


In [5]:
def convert_qty_string_to_cols(row):
    # lookup uom
    match = codelist.loc[codelist['dmd_id'] == row['dmd_id'], 'uom']
    uom = match.iloc[0] if not match.empty else None

    qty_text = row.get('quantity') if isinstance(row, dict) else row['quantity']
    if pd.isna(uom) or pd.isna(qty_text):
        return pd.Series([None, None])
   
    # patterns
    pattern_1 = re.compile(
        rf'^(\d+)\s+{re.escape(uom)}(s|\(s\))?$',
        re.IGNORECASE
    )
    pattern_2 = re.compile(
        rf'^(\d+)\s+pack(s|\(s\))?\s+of\s+(\d+)\s+{re.escape(uom)}(s|\(s\))?$',
        re.IGNORECASE
    )

    # try pattern 2 first (more specific)
    m2 = pattern_2.match(qty_text)
    if m2:
        try:
            packs = int(m2.group(1))
            per_pack = int(m2.group(3))
            total = packs * per_pack
            return pd.Series([total, uom])
        except ValueError:
            return pd.Series([None, uom])

    # then pattern 1
    m1 = pattern_1.match(qty_text)
    if m1:
        try:
            qty = int(m1.group(1))
            return pd.Series([qty, uom])
        except ValueError:
            return pd.Series([None, uom])

    # fallback if no match
    return pd.Series([None, None])


# Add calculated quantity columns
qty_table_results = qty_table_full
qty_table_results[['calculated_quantity', 'calculated_uom']] = qty_table_results.apply(
    convert_qty_string_to_cols,
    axis=1
)


# Test to see if calculated matches
qty_table_results['correct_calculation'] = (
    (qty_table_results['formatted_quantity'] == qty_table_results['calculated_quantity']) &
    (qty_table_results['formatted_units'] == qty_table_results['calculated_uom'])
)

qty_table_results

,dmd_nm,dmd_id,quantity,formatted_quantity,formatted_units,calculated_quantity,calculated_uom,correct_calculation
0,Ramipril 5mg tablets,42381711000001108,28 tablets,28,tablet,28.0,tablet,True
1,Ramipril 5mg capsules,42381611000001104,28 capsules,28,capsule,28.0,capsule,True
2,Ramipril 5mg tablets,42381711000001108,1 pack of 28 tablet(s),28,tablet,28.0,tablet,True
3,Ramipril 5mg capsules,42381611000001104,2 packs of 28 capsule(s),56,capsule,56.0,capsule,True
4,Salbutamol 100micrograms/dose inhaler CFC free,39113611000001102,600 doses,600,dose,600.0,dose,True
5,Salbutamol 100micrograms/dose inhaler CFC free,39113611000001102,1 pack of 200 dose(s),200,dose,200.0,dose,True
6,Clobetasone 0.05% cream,41894611000001102,1 pack of 100 gram(s),100,gram,NaN,None,False
7,Clobetasone 0.05% cream,41894611000001102,15 gram(s),15,gram,NaN,None,False
8,Hypromellose 0.5% eye drops,42191711000001101,10ml - 0.5%,10,ml,NaN,None,False
9,Ferrous fumarate 140mg/5ml oral solution,39108711000001103,100 millilitres,100,ml,NaN,None,False
